# 01 - Infrastructure Setup & Data Generation

This notebook walks through:
1. Creating Snowflake infrastructure (database, schema, warehouse, compute pool)
2. Generating synthetic fraud detection data
3. Exploratory data analysis on the generated dataset

**Prerequisites:** 
- Snowflake account with ACCOUNTADMIN role
- `snowflake-ml-python` installed locally
- Connection configured in `~/.snowflake/connections.toml`

In [ ]:
import sys
sys.path.insert(0, "../source")
from snowpark_session import create_snowpark_session
from config import DATABASE, SCHEMA, WAREHOUSE

session = create_snowpark_session()
print(f"Connected as: {session.get_current_user()}")
print(f"Role: {session.get_current_role()}")

## Step 1: Create Infrastructure

Run the setup SQL to create all required Snowflake objects. This creates:
- **SNOW_MLOPS_DEV** database with **ML** schema
- **SNOW_MLOPS_DEV_WH** warehouse (MEDIUM, auto-suspend 120s)
- **SNOW_MLOPS_DEV_POOL** compute pool (CPU_X64_M, 1-3 nodes)
- Internal stages for ML artifacts, DAG scripts, and job files

In [ ]:
# Create database and schema
session.sql("CREATE DATABASE IF NOT EXISTS SNOW_MLOPS_DEV").collect()
session.sql("CREATE SCHEMA IF NOT EXISTS SNOW_MLOPS_DEV.ML").collect()

# Create warehouse
session.sql("""
    CREATE WAREHOUSE IF NOT EXISTS SNOW_MLOPS_DEV_WH
        WAREHOUSE_SIZE = 'MEDIUM'
        AUTO_SUSPEND = 120
        AUTO_RESUME = TRUE
        INITIALLY_SUSPENDED = TRUE
""").collect()

# Create compute pool for ML Jobs
session.sql("""
    CREATE COMPUTE POOL IF NOT EXISTS SNOW_MLOPS_DEV_POOL
        MIN_NODES = 1
        MAX_NODES = 3
        INSTANCE_FAMILY = CPU_X64_M
        AUTO_SUSPEND_SECS = 300
        AUTO_RESUME = TRUE
""").collect()

# Create stages
session.sql("USE SCHEMA SNOW_MLOPS_DEV.ML").collect()
session.sql("CREATE STAGE IF NOT EXISTS ML_ARTIFACTS ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')").collect()
session.sql("CREATE STAGE IF NOT EXISTS DAG_STAGE ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')").collect()
session.sql("CREATE STAGE IF NOT EXISTS JOB_STAGE ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')").collect()

# Grant privileges
session.sql("GRANT EXECUTE TASK ON ACCOUNT TO ROLE ACCOUNTADMIN").collect()
session.sql("GRANT EXECUTE MANAGED TASK ON ACCOUNT TO ROLE ACCOUNTADMIN").collect()
session.sql("GRANT BIND SERVICE ENDPOINT ON ACCOUNT TO ROLE ACCOUNTADMIN").collect()

print("Infrastructure setup complete!")

## Step 2: Generate Synthetic Fraud Dataset

We generate a realistic fraud detection dataset with:
- **5,000 customers** with demographics (age, income, credit score)
- **500 merchants** across 12 categories with varying risk levels
- **100,000 transactions** over 90 days with ~3% fraud rate

Fraud patterns injected: high amounts, late-night timing, high-risk merchants, velocity anomalies.

In [ ]:
sys.path.insert(0, "../scripts")
from generate_dataset import generate_customers, generate_merchants, generate_transactions, upload_to_snowflake
import numpy as np

rng = np.random.default_rng(42)

customers = generate_customers(n=5000, rng=rng)
merchants = generate_merchants(n=500, rng=rng)
transactions = generate_transactions(customers, merchants, n=100_000, rng=rng)

print(f"Customers: {len(customers):,}")
print(f"Merchants: {len(merchants):,}")
print(f"Transactions: {len(transactions):,}")
print(f"Fraud rate: {transactions['IS_FRAUD'].mean():.2%}")

In [ ]:
# Upload to Snowflake
session.sql(f"USE WAREHOUSE {WAREHOUSE}").collect()

upload_to_snowflake(session, customers, f"{DATABASE}.{SCHEMA}.CUSTOMER_PROFILES")
upload_to_snowflake(session, merchants, f"{DATABASE}.{SCHEMA}.MERCHANT_DATA")
upload_to_snowflake(session, transactions, f"{DATABASE}.{SCHEMA}.RAW_TRANSACTIONS")

## Step 3: Exploratory Data Analysis

In [ ]:
# Quick EDA on transactions
print("=== Transaction Amount Distribution ===")
print(transactions.groupby("IS_FRAUD")["AMOUNT"].describe())
print("\n=== Fraud by Device Type ===")
print(transactions.groupby("DEVICE_TYPE")["IS_FRAUD"].mean().sort_values(ascending=False))
print("\n=== Transactions by Hour (Fraud vs Legit) ===")
transactions["HOUR"] = transactions["TIMESTAMP"].dt.hour
print(transactions.groupby(["HOUR", "IS_FRAUD"]).size().unstack(fill_value=0).head(10))

In [ ]:
# Verify data in Snowflake
for table in ["CUSTOMER_PROFILES", "MERCHANT_DATA", "RAW_TRANSACTIONS"]:
    count = session.table(f"{DATABASE}.{SCHEMA}.{table}").count()
    print(f"{table}: {count:,} rows")

session.table(f"{DATABASE}.{SCHEMA}.RAW_TRANSACTIONS").show(5)